# Part G: decision curves and pathway-level agreement (BMC revision)

Run this **after** v5 (`log2_loggenes`), Part D, Part E and Part F. It only **reads** their checkpoints and writes
everything new to `.../partG/`. Total time is about 15 to 25 minutes on a Colab CPU.

| Cell | What it does |
|---|---|
| G0-G0b | setup, data, the saved fold models and out-of-fold risks |
| G1 | **Decision curve analysis** at 12 and 24 months for the clinical network, the gated fusion network and Coxnet (out-of-fold, Breslow baseline from each training fold; net benefit with Kaplan-Meier event rates; bootstrap intervals) |
| G2 | **Hallmark pathway-level agreement** across the 324 explanations of the Part D grid, with the same one-factor and Shapley-R2 decomposition as Part F (F3), and a random-gene-set control |
| G3 | summary JSON and figures |

When it finishes, send back the `partG` folder (or `partG_results.json`, the `G*.csv` files and `fig_dca.png`).

In [ ]:
# ===== G0. Setup =====
import os, sys, subprocess, json, pickle, time, warnings, urllib.request
from itertools import combinations, product
from math import factorial
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lifelines', 'scikit-survival', 'gseapy'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')

BASE      = os.environ.get('PAPER_BASE', '/content/drive/MyDrive')
DATA_DIR  = f'{BASE}/pancreatic_paper_data'
V5_DIR    = f'{BASE}/pancreatic_paper_checkpoints_log2_loggenes'
PARTD_DIR = f'{V5_DIR}/partD'
OUT       = f'{V5_DIR}/partG'
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, KFold, GridSearchCV
from scipy.stats import spearmanr
from lifelines import KaplanMeierFitter
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.util import Surv
warnings.filterwarnings('ignore')
f32 = np.float32
N_BOOT = 1000


def load_from(folder, name):
    path = f'{folder}/{name}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None


def need(folder, name):
    obj = load_from(folder, name)
    if obj is None:
        raise FileNotFoundError(f"{name}.pkl not found in {folder}. Run v5 and Parts D-F first.")
    return obj


def save(name, obj):
    with open(f'{OUT}/{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)


def load(name):
    return load_from(OUT, name)


RESULTS = {}
print("outputs ->", OUT)

In [ ]:
# ===== G0b. Data, folds and saved models (identical to v5 / Part F) =====
df = pd.read_csv(f'{DATA_DIR}/tcga_paad_clean_log2.csv')
stage_map = {'STAGE I': 1, 'STAGE IA': 1, 'STAGE IB': 1, 'STAGE II': 2, 'STAGE IIA': 2, 'STAGE IIB': 2,
             'STAGE III': 3, 'STAGE IV': 4}
df['STAGE_ENCODED'] = df['AJCC_PATHOLOGIC_TUMOR_STAGE'].map(stage_map).fillna(2)
gene_cols = df.columns[5:1005].tolist()
CLIN_COLS = ['AGE', 'STAGE_ENCODED']
NC = len(CLIN_COLS)
N = len(df)
Xc_raw = df[CLIN_COLS].values.astype(f32)
Xg_log = df[gene_cols].values.astype(f32)
if Xg_log.max() > 100:
    Xg_log = np.log2(Xg_log + 1)
Xg_clip = np.clip(Xg_log, np.percentile(Xg_log, 1, axis=0), np.percentile(Xg_log, 99, axis=0)).astype(f32)
T = df['OS_MONTHS'].values.astype(float)
E = df['OS_STATUS'].values.astype(int)
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(Xc_raw, E))


class GatedFusion(nn.Module):
    def __init__(self, n_clin=2, n_gen=1000):
        super().__init__()
        self.clin_encoder = nn.Sequential(nn.Linear(n_clin, 32), nn.BatchNorm1d(32), nn.ReLU(),
                                          nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU())
        self.gen_encoder = nn.Sequential(nn.Linear(n_gen, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
                                         nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4))
        self.gate_clin = nn.Sequential(nn.Linear(192, 64), nn.Sigmoid())
        self.gate_gen = nn.Sequential(nn.Linear(192, 128), nn.Sigmoid())
        self.head = nn.Linear(192, 1)

    def forward(self, xc, xg):
        hc, hg = self.clin_encoder(xc), self.gen_encoder(xg)
        h = torch.cat([hc, hg], 1)
        return self.head(torch.cat([hc * self.gate_clin(h), hg * self.gate_gen(h)], 1)).squeeze(1)


GatedFusionModel = GatedFusion   # class name used when the v5 checkpoints were pickled
cv_state = need(V5_DIR, 'cv_state')
FOLD_SCALERS = cv_state['fold_scalers']
assert all(np.array_equal(np.sort(te), np.sort(t5)) for (tr, te), t5 in zip(FOLDS, cv_state['fold_test_indices']))
OOF5 = need(V5_DIR, 'extended_eval_oof_preds')       # {'Clinical MLP': {'tr': [...], 'te': [...]}, 'Gated fusion': ...}
print("Saved out-of-fold models:", list(OOF5))


def harrell_c(T_, E_, r):
    T_ = np.asarray(T_, float); E_ = np.asarray(E_).astype(bool); r = np.asarray(r, float)
    comp = ((T_[:, None] < T_[None, :]) | ((T_[:, None] == T_[None, :]) & ~E_[None, :])) & E_[:, None]
    gt = (r[:, None] > r[None, :]) & comp
    eq = (r[:, None] == r[None, :]) & comp
    return (gt.sum() + 0.5 * eq.sum()) / comp.sum()


def breslow(r_tr, T_tr, E_tr):
    c = float(np.max(r_tr)); er = np.exp(np.asarray(r_tr) - c)
    ev = np.unique(T_tr[E_tr == 1])
    dH = np.array([((T_tr == t) & (E_tr == 1)).sum() / er[T_tr >= t].sum() for t in ev])
    return ev, np.cumsum(dH), c


def surv_from_risk(bres, r, times):
    ev, H, c = bres
    idx = np.searchsorted(ev, np.asarray(times, float), side='right') - 1
    H0 = np.where(idx >= 0, H[np.clip(idx, 0, None)], 0.0)
    return np.exp(-np.outer(np.exp(np.asarray(r) - c), H0))


def fit_coxnet(X, y, seed=0):
    # identical to the v5 / Part D baseline
    path = CoxnetSurvivalAnalysis(l1_ratio=0.5, alpha_min_ratio=0.05, n_alphas=25, max_iter=100000).fit(X, y)
    gcv = GridSearchCV(CoxnetSurvivalAnalysis(l1_ratio=0.5, max_iter=100000), {'alphas': [[a] for a in path.alphas_]},
                       cv=KFold(3, shuffle=True, random_state=seed), error_score=0.5, n_jobs=1).fit(X, y)
    best = gcv.best_estimator_
    if np.count_nonzero(best.coef_) < 10:
        nz = (path.coef_ != 0).sum(0); j = int(np.argmin(np.abs(nz - 30)))
        best = CoxnetSurvivalAnalysis(l1_ratio=0.5, alphas=[path.alphas_[j]], max_iter=100000).fit(X, y)
    return best

print(f"{N} patients, {E.sum()} deaths")

In [ ]:
# ===== G1. Decision curve analysis (out-of-fold) =====
# Predicted probability of death by t = 1 - S(t | x), S from a Breslow baseline fitted on each training fold.
# Net benefit at threshold p_t: NB = P(death by t | high) * P(high) - P(alive at t | high) * P(high) * p_t / (1 - p_t),
# with event probabilities from Kaplan-Meier in the 'high' group (risk >= p_t), so censoring is handled
# (Vickers et al. 2008). Reference strategies: treat all (everyone 'high') and treat none (NB = 0).
HZ = [12, 24]
MODELS = {'Clinical network': 'Clinical MLP', 'Gated fusion network': 'Gated fusion', 'Coxnet': None}
ck = load('G1_oof_probs')
if ck is None:
    P = {m: np.full((N, len(HZ)), np.nan) for m in MODELS}
    cind = {m: [] for m in MODELS}
    for f, (tr, te) in enumerate(FOLDS):
        for m, key in MODELS.items():
            if key is not None:
                r_tr, r_te = np.asarray(OOF5[key]['tr'][f]), np.asarray(OOF5[key]['te'][f])
            else:
                sc, sg = FOLD_SCALERS[f]
                X_tr = np.hstack([sc.transform(Xc_raw[tr]), sg.transform(Xg_clip[tr])]).astype(np.float64)
                X_te = np.hstack([sc.transform(Xc_raw[te]), sg.transform(Xg_clip[te])]).astype(np.float64)
                cn = fit_coxnet(X_tr, Surv.from_arrays(E[tr].astype(bool), T[tr]), seed=f)
                r_tr, r_te = cn.predict(X_tr), cn.predict(X_te)
            P[m][te] = 1 - surv_from_risk(breslow(r_tr, T[tr], E[tr]), r_te, HZ)
            cind[m].append(harrell_c(T[te], E[te], r_te))
        print(f"  fold {f + 1}/5 done")
    ck = {'P': P, 'cind': cind}
    save('G1_oof_probs', ck)
P = ck['P']
print("Out-of-fold Harrell C (mean over folds; should match Table 3: 0.595 / 0.596 / 0.654):",
      {m: round(float(np.mean(c)), 3) for m, c in ck['cind'].items()})

THR = np.round(np.arange(0.05, 0.951, 0.01), 2)


def km_death(t_, e_, h):
    if len(t_) == 0:
        return 0.0
    return 1 - float(KaplanMeierFitter().fit(t_, e_).predict(h))


def net_benefit(p, t_, e_, h, thr):
    out = np.empty(len(thr))
    for k, pt in enumerate(thr):
        hi = p >= pt
        frac = hi.mean()
        if frac == 0:
            out[k] = 0.0
            continue
        d = km_death(t_[hi], e_[hi], h)
        out[k] = d * frac - (1 - d) * frac * pt / (1 - pt)
    return out


def treat_all(t_, e_, h, thr):
    d = km_death(t_, e_, h)
    return d - (1 - d) * thr / (1 - thr)


rows, curves = [], {}
for hi_, h in enumerate(HZ):
    curves[h] = {'Treat all': treat_all(T, E, h, THR), 'Treat none': np.zeros(len(THR))}
    for m in MODELS:
        curves[h][m] = net_benefit(P[m][:, hi_], T, E, h, THR)
    for k, pt in enumerate(THR):
        rows.append({'horizon_months': h, 'threshold': pt, **{m: curves[h][m][k] for m in curves[h]}})
G1 = pd.DataFrame(rows)
G1.to_csv(f'{OUT}/G1_net_benefit_curves.csv', index=False)

# bootstrap intervals at selected thresholds, and paired differences between models
SEL = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
ckb = load('G1_boot')
if ckb is None:
    rb = np.random.RandomState(42)
    boot = []
    for b in range(N_BOOT):
        ii = rb.randint(0, N, N)
        rec = {}
        for hi_, h in enumerate(HZ):
            rec[(h, 'Treat all')] = treat_all(T[ii], E[ii], h, np.array(SEL))
            for m in MODELS:
                rec[(h, m)] = net_benefit(P[m][ii, hi_], T[ii], E[ii], h, np.array(SEL))
        boot.append(rec)
        if (b + 1) % 200 == 0:
            print(f"  bootstrap {b + 1}/{N_BOOT}")
    ckb = boot
    save('G1_boot', ckb)
rows = []
for h in HZ:
    for k, pt in enumerate(SEL):
        r = {'horizon_months': h, 'threshold': pt}
        for m in ['Treat all'] + list(MODELS):
            v = np.array([b[(h, m)][k] for b in ckb])
            r[m] = float(G1[(G1.horizon_months == h) & (G1.threshold == pt)][m].iloc[0])
            r[f'{m} 95% CI'] = f"{np.percentile(v, 2.5):.3f} to {np.percentile(v, 97.5):.3f}"
        for a, b_ in [('Coxnet', 'Gated fusion network'), ('Gated fusion network', 'Clinical network'),
                      ('Coxnet', 'Clinical network')]:
            d = np.array([bb[(h, a)][k] - bb[(h, b_)][k] for bb in ckb])
            r[f'{a} minus {b_}'] = float(np.mean(d))
            r[f'{a} minus {b_} 95% CI'] = f"{np.percentile(d, 2.5):.3f} to {np.percentile(d, 97.5):.3f}"
        rows.append(r)
G1b = pd.DataFrame(rows)
G1b.to_csv(f'{OUT}/G1_net_benefit_selected_thresholds.csv', index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
print(G1b[['horizon_months', 'threshold', 'Treat all', 'Clinical network', 'Gated fusion network', 'Coxnet']].round(3).to_string(index=False))


def useful_range(h, m):
    # thresholds at which the model beats both treat-all and treat-none by at least 0.005
    g = G1[G1.horizon_months == h]
    ok = (g[m] > np.maximum(g['Treat all'], 0) + 0.005).values
    return [float(x) for x in g.threshold.values[ok]]


summ = {}
for h in HZ:
    ev = km_death(T, E, h)
    summ[h] = {'KM_death_prob': ev, **{m: {'thresholds_with_benefit': (min(useful_range(h, m)), max(useful_range(h, m)))
                                            if useful_range(h, m) else None,
                                            'n_thresholds_with_benefit': len(useful_range(h, m))} for m in MODELS}}
    print(f"\n{h} months: KM probability of death {ev:.3f}")
    for m in MODELS:
        print(f"  {m:22s} beats treat-all and treat-none (by >= 0.005) at {summ[h][m]['n_thresholds_with_benefit']:2d} "
              f"thresholds; range {summ[h][m]['thresholds_with_benefit']}")

# figure
COL = {'Treat all': '#8a8a85', 'Treat none': '#000000', 'Clinical network': '#2a78d6',
       'Gated fusion network': '#eb6834', 'Coxnet': '#1f9e6e'}
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
for ax, h in zip(axes, HZ):
    for m, y in curves[h].items():
        ax.plot(THR, y, color=COL[m], lw=1.2 if m.startswith('Treat') else 2.0,
                ls='--' if m == 'Treat all' else '-', label=m)
    ymax = max(np.nanmax(curves[h][m]) for m in curves[h])
    ax.set_ylim(-0.05, ymax + 0.05); ax.set_xlim(THR[0], THR[-1])
    ax.set_xlabel('Threshold probability of death'); ax.set_ylabel('Net benefit')
    ax.set_title(f'{h} months', fontsize=10); ax.spines[['top', 'right']].set_visible(False)
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_dca.png', dpi=400, bbox_inches='tight'); plt.close()
RESULTS['G1'] = {'cindex': {m: float(np.mean(c)) for m, c in ck['cind'].items()}, 'summary': {str(k): v for k, v in summ.items()},
                 'selected': G1b.to_dict('records')}

In [ ]:
# ===== G2. Hallmark pathway-level agreement across the Part D grid =====
# Each of the 324 grid explanations (36 pipelines x 2 seeds x IG / SHAP-200 x2 / SHAP-2048 / LIME) is a vector of mean
# |attribution| over the pipeline's selected genes. Genes are converted to percentile ranks within the explanation, and a
# Hallmark pathway's score is the mean percentile of its member genes present (pathways with fewer than MIN_GENES
# members present are skipped). Pairwise pathway disagreement (1 - Spearman over shared pathways; 1 - Jaccard of the
# top-5 pathways) is then decomposed exactly as in Part F (F3). Control: the same computation with random gene sets of
# the same sizes, because averaging over genes alone raises agreement.
MIN_GENES, TOP_P, N_CTRL = 3, 5, 5


def hallmark_sets():
    try:
        import gseapy
        lib = gseapy.get_library('MSigDB_Hallmark_2020', organism='Human')
        return {k: set(v) for k, v in lib.items()}
    except Exception as e:
        print("gseapy/Enrichr unavailable:", e)
    url = 'https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/h.all.v2023.2.Hs.symbols.gmt'
    txt = urllib.request.urlopen(url, timeout=120).read().decode()
    return {l.split('\t')[0]: set(l.strip().split('\t')[2:]) for l in txt.splitlines() if l.strip()}


HSETS = hallmark_sets()
print(f"{len(HSETS)} Hallmark gene sets")

GRIDP = {'transform': ['raw', 'log2'], 'selection': ['variance', 'mad', 'cox'], 'k': [500, 1000, 2000],
         'scaler': ['robust', 'standard']}
ex_rows, vecs = [], []
for v in product(*GRIDP.values()):
    p = dict(zip(GRIDP, v))
    key = f"{p['transform']}_{p['selection']}_k{p['k']}_{p['scaler']}"
    rec = load_from(f'{PARTD_DIR}/grid', key)
    if rec is None:
        continue
    names = CLIN_COLS + list(rec['genes'])
    for (s, fam, budget, run), vec in rec['expl'].items():
        ex_rows.append({**p, 'pipe': key, 'seed': s, 'family': fam, 'budget': budget, 'run': run})
        vecs.append(pd.Series(np.asarray(vec)[NC:], index=names[NC:]))
EX = pd.DataFrame(ex_rows)
PAIRS = need(PARTD_DIR, 'D4_pairs')
assert len(PAIRS) == len(EX) * (len(EX) - 1) // 2, "D4_pairs does not match the grid"
FACT = ['transform', 'selection', 'k', 'scaler', 'model seed', 'method family', 'SHAP budget', 'SHAP re-run']
print(f"{len(EX)} explanations, {len(PAIRS)} pairs")


def pathway_matrix(sets):
    names = list(sets)
    M = np.full((len(vecs), len(names)), np.nan)
    cover = np.zeros((len(vecs), len(names)), int)
    for i, v in enumerate(vecs):
        pct = v.rank(pct=True)
        for j, nm in enumerate(names):
            g = pct.index.intersection(list(sets[nm]))
            cover[i, j] = len(g)
            if len(g) >= MIN_GENES:
                M[i, j] = pct[g].mean()
    return M, cover, names


def pair_disagreement(M):
    rho = np.full(len(PAIRS), np.nan); jac = np.full(len(PAIRS), np.nan)
    I, J = PAIRS['i'].values, PAIRS['j'].values
    tops = [set(np.argsort(-np.nan_to_num(M[i], nan=-1))[:TOP_P]) for i in range(len(M))]
    for n_, (i, j) in enumerate(zip(I, J)):
        m = ~np.isnan(M[i]) & ~np.isnan(M[j])
        if m.sum() >= 10:
            rho[n_] = 1 - spearmanr(M[i, m], M[j, m]).correlation
        jac[n_] = 1 - len(tops[i] & tops[j]) / len(tops[i] | tops[j])
    return rho, jac


def r2(y, X):
    A = np.column_stack([np.ones(len(y)), X])
    beta = np.linalg.lstsq(A, y, rcond=None)[0]
    return 1 - np.sum((y - A @ beta) ** 2) / np.sum((y - y.mean()) ** 2)


def shapley_r2(y, X, p):
    cache = {}

    def R(S):
        if S not in cache:
            cache[S] = r2(y, X[:, list(S)]) if S else 0.0
        return cache[S]
    out = np.zeros(p)
    for f in range(p):
        others = [g for g in range(p) if g != f]
        for k_ in range(p):
            for S in combinations(others, k_):
                w = factorial(k_) * factorial(p - k_ - 1) / factorial(p)
                out[f] += w * (R(tuple(sorted(S + (f,)))) - R(S))
    return out, R(tuple(range(p)))


lo_ex = ((EX.family == 'SHAP') & (EX.budget == 200)).values
conv = ~lo_ex[PAIRS['i'].values] & ~lo_ex[PAIRS['j'].values]


def decompose(y, label, level):
    rows = []
    for subset, msk in [('all pairs', np.ones(len(y), bool)), ('converged only', conv)]:
        ok = msk & np.isfinite(y)
        Xf = PAIRS.loc[ok, FACT].values.astype(float)
        use = FACT if subset == 'all pairs' else FACT[:6]
        Xu = PAIRS.loc[ok, use].values.astype(float)
        sh, tot = shapley_r2(y[ok], Xu, len(use))
        single = Xf.sum(1) == 1
        for f, name in enumerate(use):
            sel = single & (PAIRS.loc[ok, name].values == 1)
            rows.append({'level': level, 'metric': label, 'pairs': subset, 'factor': name,
                         'one_factor_mean': float(y[ok][sel].mean()) if sel.any() else np.nan,
                         'n_one_factor_pairs': int(sel.sum()), 'share_of_explained_%': 100 * sh[f] / sh.sum(),
                         'total_R2': tot})
    return rows


ck = load('G2_pathways')
if ck is None:
    Mh, cover, pnames = pathway_matrix(HSETS)
    rho_h, jac_h = pair_disagreement(Mh)
    rng = np.random.RandomState(0)
    universe = sorted(set().union(*[set(v.index) for v in vecs]))
    ctrl = []
    for c in range(N_CTRL):
        # match the number of member genes PRESENT in the explained gene universe, not the nominal set size
        rs = {f'R{j}': set(rng.choice(universe, max(1, len(set(universe) & s)), replace=False))
              for j, s in enumerate(HSETS.values())}
        Mr, _, _ = pathway_matrix(rs)
        ctrl.append(pair_disagreement(Mr))
        print(f"  random-set control {c + 1}/{N_CTRL}")
    ck = {'Mh': Mh, 'cover': cover, 'pnames': pnames, 'rho_h': rho_h, 'jac_h': jac_h, 'ctrl': ctrl}
    save('G2_pathways', ck)
Mh, cover, pnames, rho_h, jac_h, ctrl = ck['Mh'], ck['cover'], ck['pnames'], ck['rho_h'], ck['jac_h'], ck['ctrl']
print(f"Pathways scored per explanation (>= {MIN_GENES} genes): median {int(np.median((~np.isnan(Mh)).sum(1)))} "
      f"of {len(pnames)} (range {(~np.isnan(Mh)).sum(1).min()}-{(~np.isnan(Mh)).sum(1).max()})")

rows = []
rows += decompose(PAIRS['d_rho'].values.astype(float), '1 - Spearman', 'gene')
rows += decompose(PAIRS['d_jac'].values.astype(float), '1 - top-20 Jaccard', 'gene')
rows += decompose(rho_h, '1 - Spearman', 'Hallmark pathway')
rows += decompose(jac_h, f'1 - top-{TOP_P} Jaccard', 'Hallmark pathway')
for c, (rr, jj) in enumerate(ctrl):
    for r in decompose(rr, '1 - Spearman', 'random gene sets'):
        r['control_rep'] = c; rows.append(r)
G2 = pd.DataFrame(rows)
G2.to_csv(f'{OUT}/G2_pathway_decomposition.csv', index=False)

# compact comparison: one-factor disagreement among converged explanations, gene vs pathway vs random sets
cmp_ = (G2[G2.pairs == 'converged only']
        .groupby(['level', 'metric', 'factor'])[['one_factor_mean', 'share_of_explained_%']].mean().round(3))
print(cmp_.to_string())

# which pathways are consistently on top? (converged explanations, log2 pipelines)
conv_ex = ~lo_ex
lg = (EX['transform'] == 'log2').values
top_counts = pd.Series(0, index=pnames, dtype=float)
for i in np.where(conv_ex)[0]:
    for j in np.argsort(-np.nan_to_num(Mh[i], nan=-1))[:TOP_P]:
        top_counts[pnames[j]] += 1
top_by = {}
for lab, msk in [('raw', ~lg), ('log2', lg)]:
    cnt = pd.Series(0, index=pnames, dtype=float)
    idx = np.where(conv_ex & msk)[0]
    for i in idx:
        for j in np.argsort(-np.nan_to_num(Mh[i], nan=-1))[:TOP_P]:
            cnt[pnames[j]] += 1
    top_by[lab] = (cnt / len(idx)).sort_values(ascending=False).head(10)
    print(f"\nMost frequent top-{TOP_P} Hallmark pathways, converged explanations, {lab} pipelines (fraction of explanations):")
    print(top_by[lab].round(2).to_string())
pd.DataFrame(top_by).to_csv(f'{OUT}/G2_top_pathways.csv')
RESULTS['G2'] = {'n_pathways': len(pnames), 'min_genes': MIN_GENES, 'top_p': TOP_P,
                 'converged_comparison': cmp_.reset_index().to_dict('records'),
                 'top_pathways': {k: v.to_dict() for k, v in top_by.items()}}

In [ ]:
# ===== G3. Summary =====
with open(f'{OUT}/partG_results.json', 'w') as fh:
    json.dump(RESULTS, fh, indent=2, default=lambda o: float(o) if np.isscalar(o) else str(o))
print(f"Saved {OUT}/partG_results.json")
print("Send back the zipped partG folder (or partG_results.json, all G*.csv files and fig_dca.png).")